<a href="https://colab.research.google.com/github/shivam91264/BDM-PROJECT/blob/main/chatbot_with_memory_and_tool_calling_1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import libraries and setup OpenAI client

In [ ]:
# Import helper function to get API key
from helper import get_openai_api_key

# Import OpenAI client
from openai import OpenAI

# Import JSON library
import json

# Get API key
openai_api_key = get_openai_api_key()

# Create OpenAI client
client = OpenAI(api_key=openai_api_key)

# Model to use
model = "gpt-4o-mini"


# This cell:
# Imports required libraries
# Gets the OpenAI API key
# Creates the OpenAI client
# Selects the model that will be used for chat.

# Cell 2 — Create memory storage

In [ ]:
# Dictionary to store conversation memory
agent_memory = {"human": "", "agent": ""}

# System prompt defining chatbot behaviour
system_prompt = (
    "you are a chatbot. You have a section of your context called [MEMORY] that contains "
    "information relevant to your conversation"
)


# Comment

# This creates a memory system for the chatbot.
# Example memory stored: -
  # human: my favorite color is blue
  # agent: acknowledged
# The [MEMORY] section is later injected into the prompt so the chatbot remembers things.

# Cell 3 — Function to save memory

In [ ]:
def core_memory_save(section: str, memory: str):
    agent_memory[section] += "\n" + memory


# This function saves information into memory.

# Example:
  # core_memory_save("human", "name is sheela")

# Memory becomes:
  # human:
  # name is sheela

# Cell 4 — Define tool metadata

In [ ]:
core_memory_save_metadata = {
    "type": "function",
    "function": {
        "name": "core_memory_save",
        "description": "Save important information about you, the agent or the human you are chatting with.",
        "parameters": {
            "type": "object",
            "properties": {
                "section": {
                    "type": "string",
                    "enum": ["human", "agent"],
                    "description": "must be either 'human' or 'agent'"
                },
                "memory": {
                    "type": "string",
                    "description": "memory to save in the section"
                }
            },
            "required": ["section", "memory"]
        }
    }
}



# This defines a tool for the LLM.

# Meaning:
# The model can call this function like:

# core_memory_save(
#  section="human",
#  memory="favorite color is blue"
# )

# This is part of function calling / tool calling in GenAI.

# Cell 5 — Extend system prompt

In [ ]:
system_prompt_os = (
    system_prompt +
    "\n you must either call a tool(core_memory_save_ or write a response to the user."
)


# This instruction tells the LLM:

# Either
  # save memory, OR
  # respond to the user

# Cell 6 — Agent step function

In [ ]:
def agent_step(user_message):
    messages = [
        {"role": "system", "content": system_prompt_os},
        {"role": "system", "content": "[MEMORY]\n" + json.dumps(agent_memory)},
        {"role": "user", "content": user_message}
    ]

    while True:
        chat_completion = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[core_memory_save_metadata]
        )

        response = chat_completion.choices[0]
        messages.append(response.message)

        if not response.message.tool_calls:
            return response.message.content

        else:
            arguments = json.loads(response.message.tool_calls[0].function.arguments)

            core_memory_save(**arguments)

            messages.append({
                "role": "tool",
                "tool_call_id": response.message.tool_calls[0].id,
                "name": "core_memory_save",
                "content": f"Updated memory: {json.dumps(agent_memory)}"
            })



# This function performs one AI step.

# Flow:

# User message
#       ↓
# LLM response
#       ↓
# Check if tool call exists
#       ↓
# Yes → save memory
# No → return response

# This is the core agent loop logic.

# Cell 7 — Run chatbot

In [ ]:
if __name__ == "__main__":

    print("Chatbot with memory started. Type 'exit' to stop.\n")

    while True:

        user_input = input("You: ")

        if user_input.lower() == "exit":
            break

        response = agent_step(user_input)

        print("Bot:", response)



# This runs the chatbot in the terminal style conversation.

# Example interaction:

# You: my favorite color is blue
# Bot: Got it! Your favorite color is blue.

# You: my name is sheela
# Bot: Nice to meet you, Sheela!

# The bot remembers information because of memory saving.